In [5]:
import requests
import time
import urllib.parse
import pandas as pd
import os
import json
from datetime import datetime, date
from collections import defaultdict

In [ ]:
# ── USER CONFIG ────────────────────────────────────────────────────────────────
STEAM_COOKIE = "YOUR_STEAM_COOKIE_HERE"

START_DATE = datetime.strptime("Aug 14 2023", "%b %d %Y")
END_DATE   = datetime.strptime("Mar 31 2026", "%b %d %Y")

LONG_FILE       = "CS2_long.parquet"         # intermediate store — do not delete
OUTPUT_FILE     = "CS2_Skin_Prices.xlsx"      # wide-format final output
CHECKPOINT_FILE = "scraper_checkpoint.json"
SLEEP_SECONDS   = 3.5                         # ~17 req/min — safe rate limit
# ───────────────────────────────────────────────────────────────────────────────

HEADERS = {
    "Cookie": f"steamLoginSecure={STEAM_COOKIE}",
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
}

# Wear tiers in display order: (column label, float_lo, float_hi)
# A skin's wear is valid when the tier range overlaps [min_float, max_float].
WEAR_TIERS = [
    ("(Factory New)",    0.00, 0.07),
    ("(Minimal Wear)",   0.07, 0.15),
    ("(Field-Tested)",   0.15, 0.38),
    ("(Well-Worn)",      0.38, 0.45),
    ("(Battle-Scarred)", 0.45, 1.00),
]
WEAR_ORDER = {label: i for i, (label, _, _) in enumerate(WEAR_TIERS)}

# Weapon categories to include — everything else is skipped
WEAPON_CATEGORIES = {
    "Pistols", "Rifles", "SMGs", "Heavy",
    "Shotguns", "Machine Guns", "Knives", "Gloves",
}

In [ ]:
def valid_wears(min_float: float, max_float: float) -> list[str]:
    """Return wear labels whose float tier overlaps [min_float, max_float]."""
    return [
        label
        for label, tier_lo, tier_hi in WEAR_TIERS
        if tier_lo < max_float and tier_hi > min_float
    ]

In [ ]:
def generate_item_list() -> list[str]:
    """
    Fetch the ByMykel community skin database, filter to weapon categories only,
    and expand each base skin into only its valid wear variants.
    Returns a list of Steam market hash names in grouped wear order.
    """
    print("Fetching skin database from ByMykel API...")
    resp = requests.get(
        "https://bymykel.github.io/CSGO-API/api/en/skins.json", timeout=30
    )
    resp.raise_for_status()
    all_items = resp.json()

    seen_base: set[str] = set()
    # base_name -> list of valid wear labels (already in FN->BS order)
    skin_wears: dict[str, list[str]] = {}

    skipped_category = 0
    skipped_no_name  = 0

    for item in all_items:
        weapon  = item.get("weapon") or {}
        pattern = item.get("pattern") or {}

        category     = weapon.get("category", "")
        weapon_name  = weapon.get("name", "").strip()
        pattern_name = pattern.get("name", "").strip()

        if category not in WEAPON_CATEGORIES:
            skipped_category += 1
            continue

        if weapon_name and pattern_name:
            base_name = f"{weapon_name} | {pattern_name}"
        elif weapon_name:
            base_name = weapon_name      # vanilla knives
        else:
            skipped_no_name += 1
            continue

        if base_name in seen_base:
            continue
        seen_base.add(base_name)

        min_f = float(item.get("min_float") or 0.00)
        max_f = float(item.get("max_float") or 1.00)
        wears = valid_wears(min_f, max_f)
        if wears:
            skin_wears[base_name] = wears

    # Build final ordered list: all wear variants of skin A, then skin B, ...
    market_names: list[str] = [
        f"{base} {wear}"
        for base, wears in sorted(skin_wears.items())
        for wear in wears
    ]

    print(
        f"Done.  {len(seen_base)} unique base skins -> "
        f"{len(market_names)} market listings to scrape.  "
        f"(Skipped {skipped_category} non-weapon, {skipped_no_name} unnamed)"
    )
    return market_names

In [ ]:
def fetch_item(item_name: str, idx: int, total: int) -> list[dict] | str:
    """
    Fetch Steam price history for one market listing.

    Returns:
      - list of row dicts  {Skin Name, Date (date obj), Average Price, Volume}
      - "RATE_LIMITED" string on HTTP 429
      - [] on any other error or no data

    Steam returns hourly points as ["Mmm DD YYYY HH: +0", price, volume].
    We slice to 11 chars to get the date string, parse it unambiguously with
    strptime, and aggregate all intra-day points into a single VWAP.
    """
    print(f"[{idx}/{total}] {item_name}")
    url = (
        "https://steamcommunity.com/market/pricehistory/"
        f"?appid=730&market_hash_name={urllib.parse.quote(item_name)}"
    )

    try:
        resp = requests.get(url, headers=HEADERS, timeout=20)
    except requests.RequestException as exc:
        print(f"  [!] Network error: {exc}")
        return []

    if resp.status_code == 429:
        print("  [!!!] RATE LIMITED - stopping.")
        return "RATE_LIMITED"

    try:
        data = resp.json()
    except ValueError:
        print(f"  [!] Non-JSON response (HTTP {resp.status_code})")
        return []

    if not data.get("success") or not data.get("prices"):
        print(f"  [!] No market data")
        return []

    daily: dict[date, dict] = defaultdict(lambda: {"vwap_num": 0.0, "volume": 0})

    for point in data["prices"]:
        raw_date_str = point[0][:11]       # "Mmm DD YYYY"
        price        = float(point[1])
        volume       = int(point[2])

        try:
            dt = datetime.strptime(raw_date_str, "%b %d %Y")
        except ValueError:
            continue

        if not (START_DATE <= dt <= END_DATE):
            continue

        day = dt.date()                    # Python date object - no tz ambiguity
        daily[day]["vwap_num"] += price * volume
        daily[day]["volume"]   += volume

    rows = []
    for day, agg in sorted(daily.items()):
        vol = agg["volume"]
        rows.append({
            "Skin Name":     item_name,
            "Date":          day,
            "Average Price": round(agg["vwap_num"] / vol, 4) if vol > 0 else None,
            "Volume":        vol,
        })

    return rows

In [ ]:
# ── Checkpoint helpers ─────────────────────────────────────────────────────────

def load_checkpoint() -> set[str]:
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            return set(json.load(f).get("completed", []))
    return set()


def save_checkpoint(completed: set[str]) -> None:
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({"completed": sorted(completed)}, f)


# ── Parquet append (long format) ───────────────────────────────────────────────

def append_to_parquet(new_rows: list[dict]) -> None:
    """
    Merge new_rows into the long-format parquet store.
    Parquet is used here because it is far faster to read/append than Excel
    when accumulating tens of thousands of rows across thousands of round-trips.
    """
    new_df = pd.DataFrame(new_rows)
    new_df["Date"] = pd.to_datetime(new_df["Date"])

    if os.path.exists(LONG_FILE):
        existing = pd.read_parquet(LONG_FILE)
        combined = pd.concat([existing, new_df], ignore_index=True)
        combined.drop_duplicates(subset=["Skin Name", "Date"], keep="last", inplace=True)
    else:
        combined = new_df

    combined.sort_values(["Skin Name", "Date"], inplace=True)
    combined.to_parquet(LONG_FILE, index=False)


# ── Wide-format Excel rebuild ──────────────────────────────────────────────────

def _wear_sort_key(col_name: str) -> tuple[str, int]:
    """
    Sort key so columns are grouped by base skin name (alphabetical),
    then ordered FN -> MW -> FT -> WW -> BS within each skin.
    """
    for label, order in WEAR_ORDER.items():
        if col_name.endswith(label):
            base = col_name[: -len(label)].strip()
            return (base, order)
    return (col_name, 99)   # fallback - shouldn't occur


def rebuild_wide_excel() -> None:
    """
    Read the full long-format parquet and write a fresh wide-format Excel:
      - Row 1:    header  (Date | Skin1 (FN) | Skin1 (MW) | ... | Skin2 (FN) | ...)
      - Column A: Date    (one row per calendar day that has ANY trade data)
      - Other cols: VWAP for that skin on that date; blank if no trades
    Skins are grouped alphabetically by base name; wears in FN->BS order.
    """
    if not os.path.exists(LONG_FILE):
        return

    long_df = pd.read_parquet(LONG_FILE)
    long_df["Date"] = pd.to_datetime(long_df["Date"]).dt.date

    # Pivot: rows = Date, columns = Skin Name, values = Average Price
    wide = long_df.pivot_table(
        index="Date",
        columns="Skin Name",
        values="Average Price",
        aggfunc="mean",    # handles any residual duplicates after merge
    )

    # Sort columns: grouped by base skin, FN->BS within each group
    wide = wide.reindex(sorted(wide.columns, key=_wear_sort_key), axis=1)

    # Sort rows chronologically
    wide.sort_index(inplace=True)

    # Reset index so Date becomes column A, remove axis name
    wide.reset_index(inplace=True)
    wide.rename_axis(None, axis=1, inplace=True)

    wide.to_excel(OUTPUT_FILE, index=False)
    print(f"  -> Excel rebuilt: {len(wide)} dates x {wide.shape[1] - 1} skin columns")

In [ ]:
def main():
    if STEAM_COOKIE == "YOUR_STEAM_COOKIE_HERE":
        print("ERROR: Paste your steamLoginSecure cookie value into STEAM_COOKIE above.")
        return

    all_items = generate_item_list()
    total     = len(all_items)
    completed = load_checkpoint()

    if completed:
        print(f"Resuming from checkpoint - {len(completed)}/{total} already done.")

    rate_limited = False

    for idx, item in enumerate(all_items, start=1):
        if item in completed:
            continue

        rows = fetch_item(item, idx, total)

        if rows == "RATE_LIMITED":
            rate_limited = True
            break

        # After every successful fetch: append to parquet, then rebuild Excel
        if rows:
            append_to_parquet(rows)
            rebuild_wide_excel()

        completed.add(item)
        save_checkpoint(completed)

        time.sleep(SLEEP_SECONDS)

    print()
    if rate_limited:
        print(
            f"Stopped - rate limited by Steam. "
            f"{len(completed)}/{total} done. Re-run this cell to resume."
        )
    else:
        print(f"Complete! {len(completed)} items scraped. Data in {OUTPUT_FILE}.")


main()